In [ ]:
import pandas as pd
import os
import glob
import numpy as np
from scipy import stats
import warnings
import ast
from sklearn.neighbors import LocalOutlierFactor  # 导入LOF算法
import matplotlib.pyplot as plt

# 设置中文字体（避免可视化乱码）
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

warnings.filterwarnings('ignore')

# ---------------------- 第一步：配置路径（仅保留AIDAS，新增基线表格路径） ----------------------
FOLDER_PATH_AIDAS = "workdir/aidascd8tdonorresultqc3"  # 仅保留AIDAS路径
# 新增：外部基线表格路径（请替换为你的实际基线CSV路径）
BASELINE_TABLE_PATH = "workdir/cd8t_baseline_summary_table.csv"
# 元信息表仅保留AIDAS相关
META_AIDAS_MALE = "workdir/aidasmale_donor_age_results/male_donor_by_age_stage.xlsx"
META_AIDAS_FEMALE = "workdir/aidasfamale_donor_age_results/female_donor_by_age_stage.xlsx"
# 输出路径（移除eQTL相关，新增基线加载标记）
OUTPUT_RAW_SUMMARY = "workdir/aidas_raw_noise_汇总统计表格_完整版1.csv"
OUTPUT_FINAL_CORRECTED = "workdir/aidas_fixed_baseline_corrected_noise.csv"
OUTPUT_LOF_PLOT = "workdir/aidas_lof_raw_noise_detection.png"
OUTPUT_GENE_WIDE_TABLE = "workdir/aidas_fixed_baseline_gene_wide_table.csv"
OUTPUT_BASELINE_TABLE = "workdir/aidas_fixed_baseline_summary_table.csv"

# ---------------------- 核心修改：更新基线列名（适配新基线表，新增95分位数列） ----------------------
# 原有核心基线列名
NEW_BASELINE_COLS = [
    'cell_baseline',  # 替换 baseline_cell_count
    'gene_cell_ratio_baseline',  # 替换 baseline_gene_cell_ratio
    'r2_fg_ratio_baseline'  # 替换 baseline_r2_fg_ratio
]

# 新增：95分位数列名（与用户提供的基线表格一致）
DELTA_95Q_COLS = [
    'cell_delta_95q',
    'gene_cell_ratio_delta_95q',
    'r2_fg_ratio_delta_95q'
]

def load_fixed_baseline_table(baseline_csv_path):
    """
    扩展修改：加载外部固定基线表格，包含3个核心基线值 + 3个delta 95分位数（预定义）
    基线表格格式要求：包含NEW_BASELINE_COLS + DELTA_95Q_COLS所有列
    """
    print("\n" + "="*60)
    print("📋 加载外部固定基线表格（适配新列名+95分位数列）")
    print("="*60)
    
    # 检查文件是否存在
    if not os.path.exists(baseline_csv_path):
        raise FileNotFoundError(f"❌ 固定基线表格不存在 → {baseline_csv_path}")
    
    # 读取基线表格
    try:
        baseline_df = pd.read_csv(baseline_csv_path, encoding='utf-8-sig')
    except UnicodeDecodeError:
        baseline_df = pd.read_csv(baseline_csv_path, encoding='gbk')
    
    # ---------------------- 修改1：校验核心基线列 + 95分位数列 ----------------------
    required_cols = NEW_BASELINE_COLS + DELTA_95Q_COLS  # 合并必需列
    missing_cols = [col for col in required_cols if col not in baseline_df.columns]
    if missing_cols:
        raise ValueError(f"❌ 基线表格缺少必需列 → {missing_cols}，要求列：{required_cols}")
    
    # ---------------------- 修改2：提取核心基线值 + 95分位数值 ----------------------
    baseline_values = {}
    # 先提取核心基线值（原有逻辑）
    for col in NEW_BASELINE_COLS:
        # 过滤非空数值
        valid_vals = baseline_df[col].dropna()
        if len(valid_vals) == 0:
            raise ValueError(f"❌ 基线列 {col} 无有效数值")
        # 取第一个有效数值
        baseline_values[col] = float(valid_vals.iloc[0])
    
    # 新增：提取95分位数值（与核心基线值处理逻辑一致）
    for col in DELTA_95Q_COLS:
        valid_vals = baseline_df[col].dropna()
        if len(valid_vals) == 0:
            raise ValueError(f"❌ 95分位数列 {col} 无有效数值")
        # 取第一个有效数值（与基线表格中global行的数值一致）
        baseline_values[col] = float(valid_vals.iloc[0])
    
    # ---------------------- 修改3：更新打印信息（包含95分位数） ----------------------
    print(f"✅ 固定基线值加载成功（含95分位数，与基线表格一致）：")
    print(f"   - 细胞数基线（cell_baseline）：{baseline_values['cell_baseline']:.6f}")
    print(f"   - 基因/细胞数比值基线（gene_cell_ratio_baseline）：{baseline_values['gene_cell_ratio_baseline']:.6f}")
    print(f"   - R2/FG比值基线（r2_fg_ratio_baseline）：{baseline_values['r2_fg_ratio_baseline']:.8f}")
    print(f"   --- 95分位数（从基线表读取，无需动态计算）---")
    print(f"   - 细胞数delta 95分位数（cell_delta_95q）：{baseline_values['cell_delta_95q']:.6f}")
    print(f"   - 基因/细胞数比值delta 95分位数（gene_cell_ratio_delta_95q）：{baseline_values['gene_cell_ratio_delta_95q']:.6f}")
    print(f"   - R2/FG比值delta 95分位数（r2_fg_ratio_delta_95q）：{baseline_values['r2_fg_ratio_delta_95q']:.8f}")
    
    return baseline_values

def detect_outliers_by_lof(df, group_col='group', value_col='raw_noise', 
                           n_neighbors_ratio=0.05,
                           min_neighbors=5, max_neighbors=50,
                           contamination=0.1):
    """
    保留原有LOF异常检测逻辑，仅适配AIDAS单组数据
    """
    print("\n" + "="*60)
    print("🔍 开始LOF异常检测（AIDAS单组，按比例动态计算邻居数）")
    print("="*60)
    
    result_df = df.copy()
    result_df['is_outlier'] = np.nan
    result_df['lof_score'] = np.nan
    outlier_stats = {}
    
    groups = result_df[group_col].unique()
    for group in groups:
        group_df = result_df[result_df[group_col] == group].copy()
        valid_data = group_df.dropna(subset=[value_col])
        
        if len(valid_data) < min_neighbors:
            print(f"⚠️ {group}组有效样本数({len(valid_data)}) < 最小邻居数({min_neighbors})，跳过LOF检测")
            outlier_stats[group] = {
                'total_samples': len(group_df),
                'valid_samples': len(valid_data),
                'outlier_count': 0,
                'outlier_ratio': 0.0
            }
            continue
        
        # 动态计算该组的n_neighbors（按比例+上下限限制）
        group_n_neighbors = int(len(valid_data) * n_neighbors_ratio)
        group_n_neighbors = max(min_neighbors, min(group_n_neighbors, max_neighbors))
        print(f"\nℹ️ {group}组：样本量={len(valid_data)} → 动态计算邻居数={group_n_neighbors}（比例={n_neighbors_ratio}）")
        
        # 准备LOF输入数据
        X = valid_data[[value_col]].values
        # 初始化LOF模型
        lof = LocalOutlierFactor(n_neighbors=group_n_neighbors, 
                                 contamination=contamination, 
                                 novelty=False)
        y_pred = lof.fit_predict(X)
        lof_scores = -lof.negative_outlier_factor_
        
        # 标记异常值
        valid_data.loc[:, 'is_outlier'] = (y_pred == -1)
        valid_data.loc[:, 'lof_score'] = lof_scores
        
        # 合并回原数据
        result_df.loc[valid_data.index, 'is_outlier'] = valid_data['is_outlier']
        result_df.loc[valid_data.index, 'lof_score'] = valid_data['lof_score']
        
        # 统计异常值信息
        outlier_count = valid_data['is_outlier'].sum()
        outlier_ratio = outlier_count / len(valid_data)
        outlier_stats[group] = {
            'total_samples': len(group_df),
            'valid_samples': len(valid_data),
            'n_neighbors_used': group_n_neighbors,
            'outlier_count': int(outlier_count),
            'outlier_ratio': float(outlier_ratio),
            'lof_score_mean': float(lof_scores.mean()),
            'lof_score_std': float(lof_scores.std()),
            'lof_score_threshold': float(np.percentile(lof_scores, 100*(1-contamination)))
        }
        
        print(f"\n📊 {group}组LOF检测结果：")
        print(f"  总样本数：{len(group_df)} | 有效样本数：{len(valid_data)}")
        print(f"  使用邻居数：{group_n_neighbors} | 异常样本数：{outlier_count} | 异常比例：{outlier_ratio:.2%}")
        print(f"  LOF得分均值±标准差：{lof_scores.mean():.4f}±{lof_scores.std():.4f}")
    
    # 全局统计
    total_valid = result_df.dropna(subset=[value_col, 'is_outlier'])
    global_outliers = total_valid['is_outlier'].sum()
    global_ratio = global_outliers / len(total_valid) if len(total_valid) > 0 else 0
    
    outlier_stats['global'] = {
        'total_samples': len(result_df),
        'valid_samples': len(total_valid),
        'outlier_count': int(global_outliers),
        'outlier_ratio': float(global_ratio)
    }
    
    print(f"\n📈 全局LOF检测统计：")
    print(f"  有效样本数：{len(total_valid)} | 异常样本数：{int(global_outliers)} | 异常比例：{global_ratio:.2%}")
    
    # 绘制LOF检测结果图
    plot_lof_results(result_df, value_col, group_col, OUTPUT_LOF_PLOT)
    
    return result_df, outlier_stats

def plot_lof_results(df, value_col, group_col, save_path):
    """保留原有LOF可视化逻辑，适配AIDAS单组数据"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # 子图1：raw_noise分布（标记异常值）
    groups = df[group_col].unique()
    colors = ['#1f77b4']  # 单组仅用一种颜色
    for i, group in enumerate(groups):
        group_df = df[df[group_col] == group].dropna(subset=[value_col, 'is_outlier'])
        if len(group_df) == 0:
            continue
        
        # 正常样本
        normal = group_df[group_df['is_outlier'] == False]
        ax1.scatter(normal.index, normal[value_col], 
                   c=colors[i], alpha=0.6, label=f'{group}组-正常', s=30)
        # 异常样本
        outlier = group_df[group_df['is_outlier'] == True]
        ax1.scatter(outlier.index, outlier[value_col], 
                   c='red', alpha=0.8, label=f'{group}组-异常', s=60, marker='x')
    
    ax1.set_xlabel('样本索引')
    ax1.set_ylabel('raw_noise值')
    ax1.set_title('LOF异常检测结果 - AIDAS组raw_noise分布')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 子图2：LOF得分分布
    for i, group in enumerate(groups):
        group_df = df[df[group_col] == group].dropna(subset=['lof_score'])
        if len(group_df) == 0:
            continue
        ax2.hist(group_df['lof_score'], bins=20, alpha=0.6, 
                label=f'{group}组', color=colors[i], edgecolor='black')
    
    # 添加异常值阈值线
    threshold = df.dropna(subset=['lof_score'])['lof_score'].quantile(0.9)
    ax2.axvline(x=threshold, color='red', linestyle='--', label=f'异常阈值(90%分位数)')
    ax2.set_xlabel('LOF得分（越大越异常）')
    ax2.set_ylabel('样本数')
    ax2.set_title('AIDAS组LOF得分分布')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"\n✅ LOF检测结果图已保存至：{os.path.abspath(save_path)}")

def process_single_donor(csv_path):
    """保留原有单个donor处理逻辑，无任何修改"""
    try:
        # 1. 提取donorid
        file_name = os.path.basename(csv_path)
        donorid = file_name.replace(".rds_gene_analysis.csv", "")
        if not donorid:
            print(f"警告：{csv_path} 无法提取donorid，跳过")
            return None
        
        # 2. 读取数据
        try:
            df = pd.read_csv(csv_path, encoding='utf-8', low_memory=False)
        except UnicodeDecodeError:
            df = pd.read_csv(csv_path, encoding='gbk', low_memory=False)
        
        # 基础校验：新增g列为必需列
        required_cols = ['fitratio', 'fdr', 'log2fitratio', 'cv2', 'u', 'g']
        missing_cols = [col for col in required_cols if col not in df.columns]
        if missing_cols:
            print(f"警告：{donorid} 缺少关键列 {missing_cols}，跳过")
            return None
        if df.empty:
            print(f"警告：{donorid} 数据为空，跳过")
            return None
        
        # ========== 关键修改：强制使用g列作为基因名列，其余逻辑完全不变 ==========
        gene_col = 'g'  # 固定使用g列，不再自动识别其他列
        # 清洗基因名称：转为字符串、去除首尾空格、大写化（保证一致性，避免重复）
        df['gene_name_clean'] = df[gene_col].astype(str).str.strip().str.upper()
        
        # 去除空值和重复基因
        df_valid_genes = df.dropna(subset=['gene_name_clean', 'log2fitratio']).copy()
        df_valid_genes = df_valid_genes.drop_duplicates(subset=['gene_name_clean'])
        
        # 转换log2fitratio为数值型
        df_valid_genes['log2fitratio'] = pd.to_numeric(df_valid_genes['log2fitratio'], errors='coerce')
        # 过滤有效log2fitratio值
        df_valid_genes = df_valid_genes.dropna(subset=['log2fitratio'])
        
        # 保存基因log2fitratio数据（字典形式：{基因名: log2fitratio值}）
        gene_log2fitratio = dict(zip(df_valid_genes['gene_name_clean'], df_valid_genes['log2fitratio']))
        
        # 3. 核心指标提取（完全保留原有逻辑，无任何改动）
        total_genes = len(df)
        
        # 总细胞数（修改：从m列提取）
        total_cells = np.nan
        if 'm' in df.columns:
            m_series = df['m'].dropna()
            if not m_series.empty:
                try:
                    m_numeric = pd.to_numeric(m_series, errors='coerce').dropna()
                    non_zero_m = m_numeric[m_numeric != 0]
                    if not non_zero_m.empty:
                        total_cells = int(non_zero_m.iloc[0])
                except Exception as e:
                    print(f"警告：{donorid} 提取总细胞数（m列）失败：{str(e)}")
        
        # adjusted R2
        adjusted_r2 = np.nan
        if 'Adjusted_R_squared' in df.columns:
            r2_series = df['Adjusted_R_squared'].dropna()
            if not r2_series.empty:
                try:
                    r2_numeric = pd.to_numeric(r2_series, errors='coerce').dropna()
                    non_zero_r2 = r2_numeric[r2_numeric != 0]
                    if not non_zero_r2.empty:
                        adjusted_r2 = non_zero_r2.iloc[0]
                except Exception as e:
                    print(f"警告：{donorid} 提取adjusted R2失败：{str(e)}")
        
        # 拟合基因数（从n列提取）
        fitting_gene_count = np.nan
        if 'n' in df.columns:
            n_series = df['n'].dropna()
            if not n_series.empty:
                try:
                    n_numeric = pd.to_numeric(n_series, errors='coerce').dropna()
                    non_zero_n = n_numeric[n_numeric != 0]
                    if not non_zero_n.empty:
                        fitting_gene_count = int(non_zero_n.iloc[0])
                except Exception as e:
                    print(f"警告：{donorid} 提取拟合基因数（n列）失败：{str(e)}")
        
        # 4. noise基因筛选（完全保留原有逻辑）
        df_valid = df.dropna(subset=['fitratio', 'fdr'])
        try:
            df_valid['fitratio'] = pd.to_numeric(df_valid['fitratio'], errors='coerce')
            df_valid['fdr'] = pd.to_numeric(df_valid['fdr'], errors='coerce')
            noise_genes = df_valid[(df_valid['fitratio'] > 1) & (df_valid['fdr'] < 0.05)].copy()
        except Exception as e:
            print(f"警告：{donorid} 筛选noise基因失败：{str(e)}")
            noise_genes = pd.DataFrame()
        noise_gene_count = len(noise_genes)
        
        # 5. log2fitratio>0.5和<0.5的noise基因数统计
        log2fitratio_gt1_count = 0
        log2fitratio_lt1_count = 0
        if noise_gene_count > 0:
            try:
                noise_genes['log2fitratio'] = pd.to_numeric(noise_genes['log2fitratio'], errors='coerce')
                log2fitratio_valid = noise_genes['log2fitratio'].dropna()
                log2fitratio_gt1_count = len(noise_genes[noise_genes['log2fitratio'] > 0.5].dropna(subset=['log2fitratio']))
                log2fitratio_lt1_count = len(noise_genes[noise_genes['log2fitratio'] < 0.5].dropna(subset=['log2fitratio']))
            except Exception as e:
                print(f"警告：{donorid} 统计log2fitratio分布失败：{str(e)}")
        
        # 6. raw_noise计算（完全保留原有逻辑）
        raw_noise = np.nan
        if noise_gene_count > 0:
            try:
                noise_genes['cv2'] = pd.to_numeric(noise_genes['cv2'], errors='coerce')
                valid_noise_genes = noise_genes[noise_genes['cv2'] < 10].dropna(subset=['cv2']).copy()
                valid_noise_count = len(valid_noise_genes)
                
                if valid_noise_count >= 500000:
                    raw_noise = valid_noise_genes.nsmallest(50, 'cv2')['cv2'].sum()
                else:
                    raw_noise = valid_noise_genes['log2fitratio'].sum() if valid_noise_count > 0 else np.nan
            except Exception as e:
                print(f"警告：{donorid} 计算raw_noise失败：{str(e)}")
        
        # 7. 其他原统计指标（完全保留原有逻辑）
        expr_max = np.nan
        expr_min = np.nan
        if noise_gene_count > 0:
            try:
                noise_genes['u'] = pd.to_numeric(noise_genes['u'], errors='coerce')
                expr_max = noise_genes['u'].max()
                expr_min = noise_genes['u'].min()
            except Exception as e:
                print(f"警告：{donorid} 统计表达量失败：{str(e)}")
        
        cv2_max = np.nan
        cv2_min = np.nan
        cv2_gt_10_count = 0
        if noise_gene_count > 0:
            try:
                noise_genes['cv2'] = pd.to_numeric(noise_genes['cv2'], errors='coerce')
                cv2_max = noise_genes['cv2'].max()
                cv2_min = noise_genes['cv2'].min()
                cv2_gt_10_count = len(noise_genes[noise_genes['cv2'] > 10].dropna(subset=['cv2']))
            except Exception as e:
                print(f"警告：{donorid} 统计cv2失败：{str(e)}")
        
        log2fitratio_mean = np.nan
        log2fitratio_25 = np.nan
        log2fitratio_50 = np.nan
        log2fitratio_75 = np.nan
        log2fitratio_top3_avg = np.nan
        if noise_gene_count > 0:
            try:
                noise_genes['log2fitratio'] = pd.to_numeric(noise_genes['log2fitratio'], errors='coerce')
                log2fit_ratio_valid = noise_genes['log2fitratio'].dropna()
                if not log2fit_ratio_valid.empty:
                    log2fitratio_mean = log2fit_ratio_valid.mean()
                    log2fitratio_25 = log2fit_ratio_valid.quantile(0.25)
                    log2fitratio_50 = log2fit_ratio_valid.quantile(0.5)
                    log2fitratio_75 = log2fit_ratio_valid.quantile(0.75)
                    top3_vals = log2fit_ratio_valid.nlargest(3)
                    if len(top3_vals) >= 1:
                        log2fitratio_top3_avg = top3_vals.mean()
            except Exception as e:
                print(f"警告：{donorid} 统计log2fitratio分位数失败：{str(e)}")
        
        # 8. 整理结果
        result = {
            'donorid': donorid,
            'raw_noise': round(raw_noise, 4) if pd.notna(raw_noise) else np.nan,
            'noise基因数': noise_gene_count,
            'log2fitratio>0.5的噪音基因数': log2fitratio_gt1_count,
            'log2fitratio<0.5的噪音基因数': log2fitratio_lt1_count,
            '总基因数': total_genes,
            '总细胞数': total_cells if pd.notna(total_cells) else np.nan,
            '拟合基因数(fitting_gene_count)': fitting_gene_count if pd.notna(fitting_gene_count) else np.nan,
            'adjusted R2': round(adjusted_r2, 4) if pd.notna(adjusted_r2) else np.nan,
            'noise基因表达量最大值': round(expr_max, 6) if pd.notna(expr_max) else np.nan,
            'noise基因表达量最小值': round(expr_min, 6) if pd.notna(expr_min) else np.nan,
            'cv2最大值': round(cv2_max, 4) if pd.notna(cv2_max) else np.nan,
            'cv2最小值': round(cv2_min, 4) if pd.notna(cv2_min) else np.nan,
            'cv2大于10的噪音基因数': cv2_gt_10_count,
            'log2fitratio平均值': round(log2fitratio_mean, 4) if pd.notna(log2fitratio_mean) else np.nan,
            'log2fitratio_25分位数': round(log2fitratio_25, 4) if pd.notna(log2fitratio_25) else np.nan,
            'log2fitratio_50分位数': round(log2fitratio_50, 4) if pd.notna(log2fitratio_50) else np.nan,
            'log2fitratio_75分位数': round(log2fitratio_75, 4) if pd.notna(log2fitratio_75) else np.nan,
            'log2fitratio_top3_avg': round(log2fitratio_top3_avg, 4) if pd.notna(log2fitratio_top3_avg) else np.nan,
            'gene_log2fitratio': gene_log2fitratio,
            '有效基因数': len(gene_log2fitratio)
        }
    
        return result
    
    except Exception as e:
        print(f"处理 {csv_path} 时出错：{str(e)}，跳过该文件")
        return None

def batch_process_folder(folder_path, group_name):
    """保留原有批量处理逻辑，仅处理AIDAS文件夹"""
    csv_files = glob.glob(os.path.join(folder_path, "**", "*.rds_gene_analysis.csv"), recursive=True)
    
    if len(csv_files) == 0:
        print(f"错误：在 {folder_path} 中未找到任何 *.rds_gene_analysis.csv 文件")
        return None
    
    print(f"\n===== 开始处理 {group_name} 组 =====")
    print(f"共发现 {len(csv_files)} 个目标文件，开始批量处理...")
    results = []
    for idx, csv_file in enumerate(csv_files, 1):
        if idx % 10 == 0:
            print(f"已处理 {idx}/{len(csv_files)} 个文件")
        
        donor_result = process_single_donor(csv_file)
        if donor_result:
            donor_result['group'] = group_name
            results.append(donor_result)
    
    final_df = pd.DataFrame(results)
    print(f"{group_name} 组处理完成！共成功提取 {len(final_df)} 个donor的指标")
    return final_df

def match_age_sex_ethnic(raw_summary_df):
    """保留原有元信息匹配逻辑，仅匹配AIDAS相关元信息"""
    print("\n" + "="*60)
    print("📋 开始匹配AIDAS组年龄、性别元信息")
    print("="*60)
    
    try:
        raw_donor_col = 'donorid'
        if raw_donor_col not in raw_summary_df.columns:
            print(f"❌ 原始汇总表缺少 {raw_donor_col} 列！无法匹配")
            return raw_summary_df
        
        raw_donors_sample = raw_summary_df[raw_donor_col].dropna().head(5).tolist()
        print(f"ℹ️  原始汇总表donor ID示例：{raw_donors_sample}")
        print(f"ℹ️  原始汇总表共有 {raw_summary_df[raw_donor_col].nunique()} 个唯一donor")
        
        raw_donors_normalized = {str(d).strip().lower(): d for d in raw_summary_df[raw_donor_col].dropna().unique()}
        
        # 仅保留AIDAS元信息配置
        meta_configs = [
            (META_AIDAS_MALE, 'aidas', 'male', 'male_donor_ids'),
            (META_AIDAS_FEMALE, 'aidas', 'female', 'female_donor_ids')
        ]
        
        meta_list = []
        for path, expected_group, expected_sex, donor_col in meta_configs:
            if not os.path.exists(path):
                print(f"⚠️  警告：{path} 文件不存在，跳过该表")
                continue
            
            if path.endswith('.xlsx') or path.endswith('.xls'):
                df = pd.read_excel(path)
            else:
                sep = '\t' if path.endswith('.tsv') else ','
                df = pd.read_csv(path, sep=sep)
            
            print(f"\n" + "-"*50)
            print(f"ℹ️  处理元信息表：{path}")
            print(f"   组别：{expected_group}，性别：{expected_sex}")
            
            required_cols = [donor_col, 'age', 'development_stage']
            missing_cols = [col for col in required_cols if col not in df.columns]
            if missing_cols:
                print(f"⚠️  缺少必需列：{missing_cols}，跳过")
                continue
            
            expanded_rows = []
            for idx, row in df.iterrows():
                donor_str = str(row[donor_col]).strip()
                if not donor_str or donor_str == 'nan':
                    continue
                
                try:
                    donor_ids = ast.literal_eval(donor_str)
                    if not isinstance(donor_ids, list):
                        donor_ids = [donor_ids]
                except:
                    donor_ids = [d.strip() for d in donor_str.replace('[', '').replace(']', '').replace("'", '').replace('"', '').split(',') if d.strip()]
                
                if idx == 0:
                    print(f"   解析后donor ID示例（前2个）：{donor_ids[:2]}")
                
                for donor_id in donor_ids:
                    donor_id_clean = str(donor_id).strip()
                    if not donor_id_clean:
                        continue
                    expanded_rows.append({
                        'donorid': donor_id_clean,
                        'age': row['age'],
                        'development_stage': row['development_stage'],
                        'group': expected_group,
                        'sex': expected_sex
                    })
            
            df_expanded = pd.DataFrame(expanded_rows)
            if df_expanded.empty:
                print(f"⚠️  该表解析后无有效donor数据，跳过")
                continue
            
            df_expanded['donorid_normalized'] = df_expanded['donorid'].astype(str).str.strip().str.lower()
            df_expanded['age'] = pd.to_numeric(df_expanded['age'], errors='coerce')
            
            df_matched = df_expanded[
                df_expanded['donorid_normalized'].isin(raw_donors_normalized.keys()) &
                (df_expanded['age'] >= 0) & (df_expanded['age'] <= 120)
            ].drop_duplicates(subset=['donorid', 'group'])
            
            print(f"   解析后总donor数：{len(df_expanded)}")
            print(f"   与原始表匹配的donor数：{len(df_matched)}")
            
            if len(df_matched) > 0:
                matched_sample = df_matched[['donorid', 'age', 'sex']].head(2)
                print(f"   匹配成功示例：\n{matched_sample}")
            
            df_matched['donorid'] = df_matched['donorid_normalized'].map(raw_donors_normalized)
            meta_list.append(df_matched[['donorid', 'group', 'sex', 'age', 'development_stage']])
        
        if meta_list:
            meta_df = pd.concat(meta_list, ignore_index=True).drop_duplicates(subset=['donorid', 'group'])
            print(f"\n" + "-"*50)
            print(f"ℹ️  所有表合并后，共匹配到 {len(meta_df)} 个唯一donor（donorid+group）")
        else:
            meta_df = pd.DataFrame()
            print(f"\n⚠️  所有表均无匹配成功的donor")
        
        merged_df = pd.merge(
            raw_summary_df,
            meta_df,
            on=['donorid', 'group'],
            how='left'
        )
        
        total_donors = len(merged_df)
        matched_donors = merged_df['age'].notna().sum()
        unmatched_donors = total_donors - matched_donors
        
        print(f"\n" + "="*50)
        print(f"✅ 元信息匹配完成！")
        print(f"📊 最终统计：")
        print(f"   总donor数：{total_donors}")
        print(f"   成功匹配元信息的donor数：{matched_donors}（{matched_donors/total_donors*100:.1f}%）")
        print(f"   未匹配的donor数：{unmatched_donors}")
        
        if matched_donors > 0:
            matched_sample = merged_df[merged_df['age'].notna()][['donorid', 'group', 'sex', 'age']].head(3)
            print(f"\n🎉 匹配成功的donor示例：")
            print(matched_sample)
        
        return merged_df
    
    except Exception as e:
        print(f"\n❌ 匹配失败：{str(e)}")
        import traceback
        traceback.print_exc()
        return raw_summary_df

def balanced_nonlinear_mapping(delta_abs, max_delta=None, slope=15, center=0.4):
    """保留原有非线性映射逻辑，无修改"""
    if max_delta is None or max_delta == 0:
        return 0
    normalized = delta_abs / max_delta
    f = 1 / (1 + np.exp(-slope * (normalized - center)))
    return np.clip(f, 0, 1)

def calculate_balanced_correction_factor(delta_series, max_delta, threshold=0.2, reverse_logic=False, 
                                         slope=15, center=0.4, min_cf=0.3, max_cf=1.7):
    """保留原有校正因子计算逻辑，无修改"""
    correction_factors = []
    for delta in delta_series:
        if pd.isna(delta):
            correction_factors.append(np.nan)
            continue
        
        if abs(delta) <= threshold:
            correction_factors.append(1.0)
            continue
        
        delta_abs = abs(delta)
        f = balanced_nonlinear_mapping(delta_abs, max_delta, slope=slope, center=center)
        
        if reverse_logic:
            a = 1 + f if delta > 0 else 1 - f
        else:
            a = 1 - f if delta > 0 else 1 + f
        
        a = np.clip(a, min_cf, max_cf)
        correction_factors.append(a)
    return correction_factors

# ---------------------- 新增：校正后noise统计函数 ----------------------
def calculate_corrected_noise_stats(df, group_col='group', raw_noise_col='raw_noise', corrected_noise_col='corrected_noise'):
    """
    计算校正后noise的详细统计信息，包括与原始raw_noise的对比
    """
    print("\n" + "="*70)
    print("📊 开始统计校正后noise详细信息（含校正前后对比）")
    print("="*70)
    
    # 初始化统计结果字典
    noise_stats = {}
    global_stats = {}
    
    # 筛选有效数据（非空、非异常）
    valid_df = df[
        (df[raw_noise_col].notna()) & 
        (df[corrected_noise_col].notna()) & 
        (df['is_outlier'] == False)
    ].copy()
    
    if len(valid_df) == 0:
        print("❌ 无有效校正后noise数据，无法进行统计")
        return noise_stats, global_stats
    
    # 按组别统计（仅AIDAS组）
    groups = valid_df[group_col].unique()
    for group in groups:
        group_df = valid_df[valid_df[group_col] == group].copy()
        group_raw_noise = group_df[raw_noise_col]
        group_corrected_noise = group_df[corrected_noise_col]
        
        # 计算描述性统计
        group_stats = {
            '有效样本数': len(group_df),
            # 原始raw_noise统计
            'raw_noise_最小值': round(group_raw_noise.min(), 4),
            'raw_noise_最大值': round(group_raw_noise.max(), 4),
            'raw_noise_均值': round(group_raw_noise.mean(), 4),
            'raw_noise_标准差': round(group_raw_noise.std(), 4),
            'raw_noise_25分位数': round(group_raw_noise.quantile(0.25), 4),
            'raw_noise_50分位数（中位数）': round(group_raw_noise.quantile(0.5), 4),
            'raw_noise_75分位数': round(group_raw_noise.quantile(0.75), 4),
            'raw_noise_变异系数(%)': round((group_raw_noise.std() / group_raw_noise.mean()) * 100, 2) if group_raw_noise.mean() != 0 else np.nan,
            # 校正后noise统计
            'corrected_noise_最小值': round(group_corrected_noise.min(), 4),
            'corrected_noise_最大值': round(group_corrected_noise.max(), 4),
            'corrected_noise_均值': round(group_corrected_noise.mean(), 4),
            'corrected_noise_标准差': round(group_corrected_noise.std(), 4),
            'corrected_noise_25分位数': round(group_corrected_noise.quantile(0.25), 4),
            'corrected_noise_50分位数（中位数）': round(group_corrected_noise.quantile(0.5), 4),
            'corrected_noise_75分位数': round(group_corrected_noise.quantile(0.75), 4),
            'corrected_noise_变异系数(%)': round((group_corrected_noise.std() / group_corrected_noise.mean()) * 100, 2) if group_corrected_noise.mean() != 0 else np.nan,
            # 校正前后对比
            '均值变化量': round(group_corrected_noise.mean() - group_raw_noise.mean(), 4),
            '均值变化率(%)': round(((group_corrected_noise.mean() - group_raw_noise.mean()) / group_raw_noise.mean()) * 100, 2) if group_raw_noise.mean() != 0 else np.nan,
            '标准差变化量': round(group_corrected_noise.std() - group_raw_noise.std(), 4),
            '标准差变化率(%)': round(((group_corrected_noise.std() - group_raw_noise.std()) / group_raw_noise.std()) * 100, 2) if group_raw_noise.std() != 0 else np.nan,
            # 相关性统计
            '校正前后皮尔逊相关系数': round(group_raw_noise.corr(group_corrected_noise), 4),
            '校正前后斯皮尔曼相关系数': round(group_raw_noise.corr(group_corrected_noise, method='spearman'), 4)
        }
        
        noise_stats[group] = group_stats
        
        # 打印组别统计结果
        print(f"\nℹ️  组别：{group}")
        print(f"   有效样本数：{group_stats['有效样本数']}")
        print(f"\n   📈 原始raw_noise统计：")
        print(f"      范围：{group_stats['raw_noise_最小值']} ~ {group_stats['raw_noise_最大值']}")
        print(f"      均值±标准差：{group_stats['raw_noise_均值']} ± {group_stats['raw_noise_标准差']}")
        print(f"      中位数（50分位数）：{group_stats['raw_noise_50分位数（中位数）']}")
        print(f"      变异系数：{group_stats['raw_noise_变异系数(%)']}%")
        print(f"\n   📉 校正后corrected_noise统计：")
        print(f"      范围：{group_stats['corrected_noise_最小值']} ~ {group_stats['corrected_noise_最大值']}")
        print(f"      均值±标准差：{group_stats['corrected_noise_均值']} ± {group_stats['corrected_noise_标准差']}")
        print(f"      中位数（50分位数）：{group_stats['corrected_noise_50分位数（中位数）']}")
        print(f"      变异系数：{group_stats['corrected_noise_变异系数(%)']}%")
        print(f"\n   🔍 校正前后对比：")
        print(f"      均值变化：{group_stats['均值变化量']}（{group_stats['均值变化率(%)']}%）")
        print(f"      标准差变化：{group_stats['标准差变化量']}（{group_stats['标准差变化率(%)']}%）")
        print(f"      皮尔逊相关系数：{group_stats['校正前后皮尔逊相关系数']}")
        print(f"      斯皮尔曼相关系数：{group_stats['校正前后斯皮尔曼相关系数']}")
    
    # 计算全局统计
    global_raw_noise = valid_df[raw_noise_col]
    global_corrected_noise = valid_df[corrected_noise_col]
    global_stats = {
        '全局有效样本数': len(valid_df),
        '全局_raw_noise_均值': round(global_raw_noise.mean(), 4),
        '全局_corrected_noise_均值': round(global_corrected_noise.mean(), 4),
        '全局_均值变化率(%)': round(((global_corrected_noise.mean() - global_raw_noise.mean()) / global_raw_noise.mean()) * 100, 2) if global_raw_noise.mean() != 0 else np.nan,
        '全局_校正前后皮尔逊相关系数': round(global_raw_noise.corr(global_corrected_noise), 4)
    }
    
    # 打印全局统计结果
    print(f"\n" + "-"*70)
    print(f"📊 全局校正后noise统计（所有有效组别合并）")
    print(f"   全局有效样本数：{global_stats['全局有效样本数']}")
    print(f"   全局raw_noise均值：{global_stats['全局_raw_noise_均值']}")
    print(f"   全局corrected_noise均值：{global_stats['全局_corrected_noise_均值']}")
    print(f"   全局均值变化率：{global_stats['全局_均值变化率(%)']}%")
    print(f"   全局校正前后皮尔逊相关系数：{global_stats['全局_校正前后皮尔逊相关系数']}")
    print(f"\n✅ 校正后noise统计完成！")
    
    return noise_stats, global_stats

def calculate_corrected_noise(merged_df, fixed_baseline_values,  # 传入固定基线值（含95分位数）
                              cell_slope=15, cell_center=0.4, cell_threshold=0, cell_min_cf=0.3, cell_max_cf=1.7,
                              gene_cell_slope=15, gene_cell_center=0.4, gene_cell_threshold=2, gene_cell_min_cf=0.3, gene_cell_max_cf=1.7,
                              r2_fg_slope=10, r2_fg_center=0.5, r2_fg_threshold=0, r2_fg_min_cf=0.3, r2_fg_max_cf=1.7,
                              spearman_min=0.1, spearman_max=0.9):
    """
    核心修改：替换delta 95分位数的获取方式（从基线表读取，不再动态计算）
    适配新基线列名+95分位数列
    """
    print("\n" + "="*60)
    print("📈 开始基于固定基线值计算校正后noise（含95分位数从基线表读取）")
    print("="*60)
    
    # 仅使用非异常样本进行校正计算
    merged_df_clean = merged_df[merged_df['is_outlier'] == False].copy()
    print(f"\n⚠️  仅使用LOF清理后的非异常样本进行校正计算：")
    print(f"  原始样本数：{len(merged_df)}")
    print(f"  非异常样本数：{len(merged_df_clean)}")
    
    if len(merged_df_clean) == 0:
        print("❌ 无有效非异常数据，无法计算校正因子")
        merged_df['corrected_noise'] = np.nan
        merged_df['log2fitratio_top3_avg_norm'] = np.nan
        merged_df['cell_correction_factor'] = np.nan
        merged_df['gene_cell_ratio_correction_factor'] = np.nan
        merged_df['r2_fg_ratio_correction_factor'] = np.nan
        merged_df['p1_spearman'] = np.nan
        merged_df['p1_clipped'] = np.nan
        merged_df['total_correction_factor'] = np.nan
        return merged_df, {}, {}
    
    print(f"ℹ️  AIDAS组非异常样本数：{len(merged_df_clean)}，开始校正计算")
    
    # ========== 修改1：细胞数校正因子（95分位数从基线表读取） ==========
    baseline_cells = fixed_baseline_values['cell_baseline']
    print(f"\n【细胞数校正因子计算（使用固定基线+预定义95分位数）】")
    print(f"⚠️  配置：1.阈值={cell_threshold} 2.非线性映射斜率={cell_slope} 中心点={cell_center} 3.校正范围={cell_min_cf}-{cell_max_cf}")
    print(f"- 固定细胞数基线（cell_baseline）：{baseline_cells:.6f}")
    
    # 计算每个个体的delta（原有逻辑不变）
    merged_df_clean['cell_delta'] = merged_df_clean['总细胞数'] - baseline_cells
    
    # 核心修改：从基线值字典读取95分位数，不再动态计算
    max_cell_delta = fixed_baseline_values['cell_delta_95q']
    print(f"- 细胞数delta 95分位数（从基线表读取，无需动态计算）：{max_cell_delta:.6f}（与基线表格global行一致）")
    
    # 计算平衡版细胞数校正因子（原有逻辑不变）
    merged_df_clean['cell_correction_factor'] = calculate_balanced_correction_factor(
        merged_df_clean['cell_delta'], max_cell_delta, 
        threshold=cell_threshold, reverse_logic=False,
        slope=cell_slope, center=cell_center,
        min_cf=cell_min_cf, max_cf=cell_max_cf
    )
    
    # 打印细胞校正因子统计
    valid_cf = merged_df_clean['cell_correction_factor'].dropna()
    if len(valid_cf) > 0:
        print(f"- 细胞数校正因子范围：{valid_cf.min():.4f} ~ {valid_cf.max():.4f}")
        print(f"- 细胞数校正因子均值±标准差：{valid_cf.mean():.4f}±{valid_cf.std():.4f}")
        print(f"- 偏离1的样本数：{len(valid_cf[valid_cf != 1.0])}")
    
    # ========== 修改2：总基因数/细胞数比值校正因子（95分位数从基线表读取） ==========
    baseline_gene_cell_ratio = fixed_baseline_values['gene_cell_ratio_baseline']
    print(f"\n【总基因数/细胞数比值校正因子计算（使用固定基线+预定义95分位数）】")
    print(f"⚠️  配置：1.阈值={gene_cell_threshold} 2.非线性映射斜率={gene_cell_slope} 中心点={gene_cell_center} 3.校正范围={gene_cell_min_cf}-{gene_cell_max_cf}")
    print(f"- 固定总基因数/细胞数比值基线（gene_cell_ratio_baseline）：{baseline_gene_cell_ratio:.6f}")
    
    # 计算每个个体的总基因数/细胞数比值（原有逻辑不变）
    merged_df_clean['gene_cell_ratio'] = merged_df_clean['总基因数'] / merged_df_clean['总细胞数']
    
    # 计算每个个体的delta（原有逻辑不变）
    merged_df_clean['gene_cell_ratio_delta'] = merged_df_clean['gene_cell_ratio'] - baseline_gene_cell_ratio
    
    # 核心修改：从基线值字典读取95分位数
    max_gene_cell_delta = fixed_baseline_values['gene_cell_ratio_delta_95q']
    print(f"- 基因/细胞数比值delta 95分位数（从基线表读取）：{max_gene_cell_delta:.6f}（与基线表格global行一致）")
    
    # 计算平衡版总基因数比值校正因子（原有逻辑不变）
    merged_df_clean['gene_cell_ratio_correction_factor'] = calculate_balanced_correction_factor(
        merged_df_clean['gene_cell_ratio_delta'], max_gene_cell_delta, 
        threshold=gene_cell_threshold, reverse_logic=False,
        slope=gene_cell_slope, center=gene_cell_center,
        min_cf=gene_cell_min_cf, max_cf=gene_cell_max_cf
    )
    
    # 打印总基因数比值校正因子统计
    valid_gene_cf = merged_df_clean['gene_cell_ratio_correction_factor'].dropna()
    if len(valid_gene_cf) > 0:
        uncorrected = len(valid_gene_cf[valid_gene_cf == 1.0])
        corrected = len(valid_gene_cf) - uncorrected
        print(f"- 总基因数比值校正因子范围：{valid_gene_cf.min():.4f} ~ {valid_gene_cf.max():.4f}")
        print(f"- 总基因数比值校正因子均值±标准差：{valid_gene_cf.mean():.4f}±{valid_gene_cf.std():.4f}")
        print(f"- 有效样本数：{len(valid_gene_cf)}（未修正：{uncorrected}，修正：{corrected}）")
    
    # ========== 修改3：R2/fittinggenecount比值校正因子（95分位数从基线表读取） ==========
    baseline_r2_fg_ratio = fixed_baseline_values['r2_fg_ratio_baseline']
    print(f"\n【adjusted R2/fittinggenecount比值校正因子计算（使用固定基线+预定义95分位数，反转逻辑）】")
    print(f"⚠️  配置：1.阈值={r2_fg_threshold} 2.非线性映射斜率={r2_fg_slope} 中心点={r2_fg_center} 3.校正范围={r2_fg_min_cf}-{r2_fg_max_cf}")
    print(f"- 固定R2/fittinggenecount比值基线（r2_fg_ratio_baseline）：{baseline_r2_fg_ratio:.8f}")
    
    # 定义比值计算函数（原有逻辑不变）
    def calc_r2_fg_ratio(row):
        adj_r2 = row['adjusted R2']
        fg_count = row['拟合基因数(fitting_gene_count)']
        if pd.isna(adj_r2) or pd.isna(fg_count) or fg_count == 0:
            return np.nan
        return adj_r2 / fg_count
    
    merged_df_clean['r2_fg_ratio'] = merged_df_clean.apply(calc_r2_fg_ratio, axis=1)
    
    # 计算每个个体的delta（原有逻辑不变）
    merged_df_clean['r2_fg_ratio_delta'] = merged_df_clean['r2_fg_ratio'] - baseline_r2_fg_ratio
    
    # 核心修改：从基线值字典读取95分位数
    max_r2_fg_delta = fixed_baseline_values['r2_fg_ratio_delta_95q']
    print(f"- R2/fg比值delta 95分位数（从基线表读取）：{max_r2_fg_delta:.8f}（与基线表格global行一致）")
    
    # 计算反转逻辑的校正因子（原有逻辑不变）
    merged_df_clean['r2_fg_ratio_correction_factor'] = calculate_balanced_correction_factor(
        merged_df_clean['r2_fg_ratio_delta'], max_r2_fg_delta,
        threshold=r2_fg_threshold, reverse_logic=True,
        slope=r2_fg_slope, center=r2_fg_center,
        min_cf=r2_fg_min_cf, max_cf=r2_fg_max_cf
    )
    
    # 打印R2/fg比值校正因子统计
    valid_r2_cf = merged_df_clean['r2_fg_ratio_correction_factor'].dropna()
    if len(valid_r2_cf) > 0:
        uncorrected = len(valid_r2_cf[valid_r2_cf == 1.0])
        corrected = len(valid_r2_cf) - uncorrected
        print(f"- R2/fg比值校正因子范围：{valid_r2_cf.min():.8f} ~ {valid_r2_cf.max():.8f}")
        print(f"- R2/fg比值校正因子均值±标准差：{valid_r2_cf.mean():.8f}±{valid_r2_cf.std():.8f}")
        print(f"- 有效样本数：{len(valid_r2_cf)}（未修正：{uncorrected}，修正：{corrected}）")
    
    # ========== 新增：计算p1（拟合基因数与raw_noise的斯皮尔曼相关系数）（原有逻辑不变） ==========
    print(f"\n【计算p1：拟合基因数与raw_noise的斯皮尔曼相关系数（用于校正因子）】")
    print(f"⚠️  配置：斯皮尔曼系数截断范围={spearman_min}-{spearman_max}")
    
    # 筛选有效样本（拟合基因数、raw_noise均非空，非异常样本）
    aidas_valid_corr = merged_df_clean.dropna(subset=['拟合基因数(fitting_gene_count)', 'raw_noise'])
    aidas_valid_corr = aidas_valid_corr[aidas_valid_corr['拟合基因数(fitting_gene_count)'] > 0]  # 排除无效拟合基因数
    
    if len(aidas_valid_corr) >= 2:  # 至少2个样本才计算相关系数
        # 计算斯皮尔曼相关系数
        p1, p1_pvalue = stats.spearmanr(
            aidas_valid_corr['拟合基因数(fitting_gene_count)'],
            aidas_valid_corr['raw_noise']
        )
        # 确保系数在合理范围（-1到1），避免极端值
        p1 = np.clip(p1, -1, 1)
        # 应用上下限阈值截断，得到用于校正的p1_clipped
        p1_clipped = np.clip(p1, spearman_min, spearman_max)
        
        # 打印p1计算结果
        print(f"- 有效样本数（拟合基因数+raw_noise非空）：{len(aidas_valid_corr)}")
        print(f"- 原始斯皮尔曼相关系数（p1）：{p1:.4f}（p值：{p1_pvalue:.6f}）")
        print(f"- 截断后p1（p1_clipped）：{p1_clipped:.4f}（截断范围：{spearman_min}-{spearman_max}）")
    else:
        # 样本不足时默认系数为1（不影响原有校正逻辑，无额外修正作用）
        p1 = 1.0
        p1_clipped = 1.0
        p1_pvalue = np.nan
        print(f"- 有效样本数不足2个，无法计算斯皮尔曼相关系数，默认p1=1.0，p1_clipped=1.0")
    
    # 将p1和p1_clipped写入数据框（所有非异常样本共享同一p1值）
    merged_df_clean['p1_spearman'] = p1
    merged_df_clean['p1_clipped'] = p1_clipped
    
    # ========== 保留原有：u值归一化逻辑 ==========
    print(f"\n【log2fitratio前三个最大值的平均值（u）统计】")
    u_vals = merged_df_clean['log2fitratio_top3_avg'].dropna()
    u_mean = u_vals.mean() if len(u_vals) > 0 else 0
    print(f"- AIDAS组u的均值：{u_mean:.4f}（有效样本数：{len(u_vals)}）")
    merged_df_clean['log2fitratio_top3_avg_norm'] = merged_df_clean['log2fitratio_top3_avg'] / u_mean if u_mean != 0 else np.nan
    
    # ========== 保留原有：总校正因子计算（追加p1_clipped的乘法整合） ==========
    print(f"\n【计算最终校正后的noise值（含p1_clipped校正）】")
    # 合并校正因子到原始数据框（所有字段均已存在，无KeyError）
    merge_cols = [
        'donorid', 'group', 'log2fitratio_top3_avg_norm',
        'cell_correction_factor', 'gene_cell_ratio_correction_factor',
        'r2_fg_ratio_correction_factor', 'p1_spearman', 'p1_clipped'
    ]
    
    merged_df = pd.merge(
        merged_df,
        merged_df_clean[merge_cols],
        on=['donorid', 'group'],
        how='left'
    )
    
    # 计算总校正因子（所有校正因子相乘，新增p1_clipped）
    valid_mask = (merged_df['is_outlier'] == False) & (merged_df['raw_noise'].notna())
    valid_df = merged_df[valid_mask].copy()
    
    if len(valid_df) > 0:
        valid_df['total_correction_factor'] = (
            valid_df['cell_correction_factor'].fillna(1.0) *
            valid_df['gene_cell_ratio_correction_factor'].fillna(1.0) *
            valid_df['r2_fg_ratio_correction_factor'].fillna(1.0) *
            valid_df['p1_clipped'].fillna(1.0)  # 新增：整合p1_clipped到总校正因子
        )
        
        # 计算校正后noise（总校正因子已包含p1_clipped）
        valid_df['corrected_noise'] = valid_df['raw_noise'] * valid_df['total_correction_factor']
        
        # 合并回原始数据框
        merged_df.loc[valid_df.index, 'corrected_noise'] = valid_df['corrected_noise']
        merged_df.loc[valid_df.index, 'total_correction_factor'] = valid_df['total_correction_factor']
        
        # 打印校正结果统计（追加p1相关说明）
        print(f"- 非异常样本校正完成：有效样本数={len(valid_df)}")
        print(f"- 校正前raw_noise范围：{valid_df['raw_noise'].min():.4f} ~ {valid_df['raw_noise'].max():.4f}")
        print(f"- 校正后noise范围：{valid_df['corrected_noise'].min():.4f} ~ {valid_df['corrected_noise'].max():.4f}")
        print(f"- 总校正因子范围（含p1_clipped）：{valid_df['total_correction_factor'].min():.4f} ~ {valid_df['total_correction_factor'].max():.4f}")
        print(f"- 用于校正的p1_clipped值：{p1_clipped:.4f}")
    else:
        print(f"- 无有效非异常样本，无法计算校正后noise")
    
    # 异常样本的相关字段设为NaN
    merged_df.loc[merged_df['is_outlier'] == True, ['corrected_noise', 'p1_spearman', 'p1_clipped', 'total_correction_factor']] = np.nan
    
    print(f"\n✅ 基于固定基线的noise校正完成（含95分位数从基线表读取）！")
    print(f"- 原始raw_noise有效样本数：{merged_df['raw_noise'].notna().sum()}")
    print(f"- 校正后noise有效样本数：{merged_df['corrected_noise'].notna().sum()}")
    print(f"- 异常样本数（校正后设为NaN）：{merged_df['is_outlier'].sum()}")
    
    # ========== 保留原有：调用校正后noise统计函数 ==========
    noise_stats, global_noise_stats = calculate_corrected_noise_stats(merged_df)
    
    return merged_df, noise_stats, global_noise_stats

def build_gene_wide_table(final_df):
    """保留原有基因宽表生成逻辑，适配AIDAS单组数据"""
    print("\n" + "="*60)
    print("📊 开始生成AIDAS组基因水平校正宽表")
    print("="*60)
    
    # 1. 筛选有效数据（非异常样本 + 有总校正因子 + 有基因log2fitratio数据）
    valid_donors = final_df[
        (final_df['is_outlier'] == False) &
        (final_df['total_correction_factor'].notna()) &
        (final_df['gene_log2fitratio'].notna())
    ].copy()
    
    if len(valid_donors) == 0:
        print("❌ 无有效捐赠者数据，无法生成宽表")
        return pd.DataFrame()
    
    print(f"ℹ️  有效捐赠者数（非异常+有校正因子+有基因数据）：{len(valid_donors)}")
    
    # 2. 提取所有唯一基因（合并所有捐赠者的基因列表）
    all_genes = set()
    for gene_dict in valid_donors['gene_log2fitratio']:
        if isinstance(gene_dict, dict):
            all_genes.update(gene_dict.keys())
    all_genes = sorted(list(all_genes))
    print(f"ℹ️  所有捐赠者合并后唯一基因数：{len(all_genes)}")
    
    # 3. 构建宽表数据
    wide_table_data = []
    for idx, row in valid_donors.iterrows():
        donor_info = {
            'donorid': row['donorid'],
            'group': row['group'],
            'sex': row['sex'] if pd.notna(row['sex']) else 'unknown',
            'age': row['age'] if pd.notna(row['age']) else np.nan,
            'total_correction_factor': row['total_correction_factor']
        }
        
        # 提取该捐赠者的基因log2fitratio数据
        gene_dict = row['gene_log2fitratio']
        correction_factor = row['total_correction_factor']
        
        # 对每个基因应用校正（保留原有逻辑：原始log2fitratio × 总校正因子）
        for gene in all_genes:
            raw_log2 = gene_dict.get(gene, np.nan)
            if pd.isna(raw_log2):
                donor_info[gene] = np.nan
                continue
            
            # 完全保留原有校正逻辑
            corrected_log2 = raw_log2 * correction_factor
            
            # 空值处理（与原有逻辑一致）
            if corrected_log2 == 0 or pd.isna(corrected_log2):
                donor_info[gene] = np.nan
            else:
                donor_info[gene] = round(corrected_log2, 6)
        
        wide_table_data.append(donor_info)
    
    # 4. 转换为DataFrame并清理
    wide_table = pd.DataFrame(wide_table_data)
    
    # 过滤全空基因列（无任何有效校正值的基因）
    gene_cols = [col for col in wide_table.columns if col in all_genes]
    non_empty_gene_cols = [col for col in gene_cols if wide_table[col].notna().sum() > 0]
    wide_table = wide_table[['donorid', 'group', 'sex', 'age', 'total_correction_factor'] + non_empty_gene_cols]
    
    print(f"✅ 宽表生成完成！")
    print(f"   - 宽表行数（捐赠者数）：{len(wide_table)}")
    print(f"   - 宽表列数（信息列+有效基因列）：{len(wide_table.columns)}")
    print(f"   - 有效基因列数（非全空）：{len(non_empty_gene_cols)}")
    
    return wide_table

def build_baseline_table(final_df, fixed_baseline_values, noise_stats=None, global_noise_stats=None):
    """
    扩展修改：新增95分位数列的记录，与输入基线表格保持一致
    """
    print("\n" + "="*60)
    print("📊 开始生成AIDAS组固定基线统计汇总表（含95分位数，与输入表格一致）")
    print("="*60)
    
    # 初始化统计信息
    if noise_stats is None:
        noise_stats = {}
    if global_noise_stats is None:
        global_noise_stats = {}
    
    # 1. 筛选非异常样本
    clean_df = final_df[final_df['is_outlier'] == False].copy()
    if len(clean_df) == 0:
        print("❌ 无有效非异常样本，无法生成基线表")
        return pd.DataFrame()
    
    # 2. 构建基线数据（新增95分位数记录）
    baseline_data = []
    group_stats = {'group': 'global'}  # 与输入基线表格的group名称一致
    valid_count = len(clean_df)
    group_stats['valid_sample_count'] = valid_count
    
    # ========== 填入核心基线值 + 95分位数值（与输入基线表格一致） ==========
    group_stats['cell_baseline'] = round(fixed_baseline_values['cell_baseline'], 6)
    group_stats['gene_cell_ratio_baseline'] = round(fixed_baseline_values['gene_cell_ratio_baseline'], 6)
    group_stats['r2_fg_ratio_baseline'] = round(fixed_baseline_values['r2_fg_ratio_baseline'], 8)
    # 新增：填入95分位数
    group_stats['cell_delta_95q'] = round(fixed_baseline_values['cell_delta_95q'], 6)
    group_stats['gene_cell_ratio_delta_95q'] = round(fixed_baseline_values['gene_cell_ratio_delta_95q'], 6)
    group_stats['r2_fg_ratio_delta_95q'] = round(fixed_baseline_values['r2_fg_ratio_delta_95q'], 8)
    
    # 填入AIDAS组统计值
    u_vals = clean_df['log2fitratio_top3_avg'].dropna()
    group_stats['u_mean'] = round(u_vals.mean(), 4) if len(u_vals) > 0 else np.nan
    
    # 填入校正后noise关键统计（从noise_stats中提取）
    aidas_noise_stats = noise_stats.get('aidas', {})
    group_stats['raw_noise_mean'] = aidas_noise_stats.get('raw_noise_均值', np.nan)
    group_stats['corrected_noise_mean'] = aidas_noise_stats.get('corrected_noise_均值', np.nan)
    group_stats['corrected_noise_median'] = aidas_noise_stats.get('corrected_noise_50分位数（中位数）', np.nan)
    group_stats['mean_change_rate(%)'] = aidas_noise_stats.get('均值变化率(%)', np.nan)
    group_stats['pearson_correlation'] = aidas_noise_stats.get('校正前后皮尔逊相关系数', np.nan)
    
    # 填入全局统计值
    group_stats['global_valid_samples'] = global_noise_stats.get('全局有效样本数', np.nan)
    group_stats['global_mean_change_rate(%)'] = global_noise_stats.get('全局_均值变化率(%)', np.nan)
    
    baseline_data.append(group_stats)
    
    # 3. 转换为DataFrame
    baseline_table = pd.DataFrame(baseline_data)
    
    print(f"✅ 固定基线表生成完成（与输入表格格式一致，含95分位数）！")
    print(f"   - 基线表行数：{len(baseline_table)}")
    print(f"   - 基线表列数：{len(baseline_table.columns)}")
    print(f"\n📋 固定基线表预览（含95分位数）：")
    print(baseline_table.round(4))
    
    return baseline_table

def main():
    """
    主函数：一键运行AIDAS组固定基线校正流程（含95分位数从基线表读取）
    """
    print("="*80)
    print("🎯 开始执行AIDAS组固定基线noise校正流程（含95分位数与基线表格一致）")
    print("="*80)
    
    # ---------------------- 步骤0：加载外部固定基线表格（含95分位数） ----------------------
    try:
        fixed_baseline_values = load_fixed_baseline_table(BASELINE_TABLE_PATH)
    except Exception as e:
        print(f"❌ 加载固定基线表格失败：{e}")
        return
    
    # ---------------------- 步骤1：批量处理AIDAS文件夹数据 ----------------------
    print("\n📌 步骤1：批量处理AIDAS文件夹")
    df_aidas_raw = batch_process_folder(FOLDER_PATH_AIDAS, group_name='aidas')
    
    # 检查数据是否加载成功
    if df_aidas_raw is None:
        print("❌ AIDAS数据加载失败！请检查文件夹路径和文件格式")
        return
    
    print(f"\n✅ AIDAS原始数据处理完成：共 {len(df_aidas_raw)} 个donor")
    
    # ---------------------- 步骤2：匹配年龄/性别元信息 ----------------------
    print("\n📌 步骤2：匹配AIDAS组年龄、性别、发育阶段元信息")
    df_merged = match_age_sex_ethnic(df_aidas_raw)
    
    # ---------------------- 步骤3：LOF异常检测（raw_noise） ----------------------
    print("\n📌 步骤3：LOF算法检测AIDAS组raw_noise异常值")
    df_merged_with_outlier, outlier_stats = detect_outliers_by_lof(
        df_merged,
        group_col='group',
        value_col='raw_noise',
        n_neighbors_ratio=0.01,
        min_neighbors=1,
        max_neighbors=50,
        contamination=0.1
    )
    
    # ---------------------- 步骤4：基于固定基线计算校正后noise（含95分位数） ----------------------
    print("\n📌 步骤4：基于外部固定基线计算AIDAS组校正后noise（含95分位数）")
    df_final, noise_stats, global_noise_stats = calculate_corrected_noise(
        df_merged_with_outlier,
        fixed_baseline_values,  # 传入含95分位数的基线值
        cell_slope=15, cell_center=0.4, cell_threshold=0, cell_min_cf=0.3, cell_max_cf=1.7,
        gene_cell_slope=15, gene_cell_center=0.4, gene_cell_threshold=2, gene_cell_min_cf=0.4, gene_cell_max_cf=1.6,
        r2_fg_slope=10, r2_fg_center=0.5, r2_fg_threshold=0, r2_fg_min_cf=0.6, r2_fg_max_cf=1.4,
        spearman_min=0.4, spearman_max=1
    )
    
    # ---------------------- 步骤5：生成基因宽表和固定基线表（含95分位数） ----------------------
    print("\n📌 步骤5：生成AIDAS组基因水平校正宽表和固定基线统计汇总表")
    # 生成宽表
    gene_wide_table = build_gene_wide_table(df_final)
    # 生成基线表（含95分位数，与输入表格一致）
    baseline_table = build_baseline_table(df_final, fixed_baseline_values, noise_stats, global_noise_stats)
    
    # ---------------------- 步骤6：保存所有结果文件 ----------------------
    print("\n📌 步骤6：保存所有分析结果文件")
    # 保存原始汇总表
    df_aidas_raw.to_csv(OUTPUT_RAW_SUMMARY, index=False, encoding='utf-8-sig')
    print(f"✅ AIDAS原始汇总表已保存：{os.path.abspath(OUTPUT_RAW_SUMMARY)}")
    
    # 保存最终校正结果表
    df_final.to_csv(OUTPUT_FINAL_CORRECTED, index=False, encoding='utf-8-sig')
    print(f"✅ AIDAS固定基线校正结果表（含corrected_noise）已保存：{os.path.abspath(OUTPUT_FINAL_CORRECTED)}")
    
    # 保存基因宽表
    if not gene_wide_table.empty:
        gene_wide_table.to_csv(OUTPUT_GENE_WIDE_TABLE, index=False, encoding='utf-8-sig')
        print(f"✅ AIDAS固定基线基因宽表已保存：{os.path.abspath(OUTPUT_GENE_WIDE_TABLE)}")
    
    # 保存基线表（含95分位数，与输入表格一致）
    if not baseline_table.empty:
        baseline_table.to_csv(OUTPUT_BASELINE_TABLE, index=False, encoding='utf-8-sig')
        print(f"✅ AIDAS固定基线统计汇总表（含95分位数）已保存：{os.path.abspath(OUTPUT_BASELINE_TABLE)}")
    
    # ---------------------- 流程结束 ----------------------
    print("\n" + "="*80)
    print("🎉 AIDAS组固定基线noise校正流程（含95分位数与基线表格一致）执行完成！")
    print(f"📊 关键结果汇总：")
    print(f"   - 原始样本数：{len(df_aidas_raw)}")
    print(f"   - 匹配元信息样本数：{df_merged['age'].notna().sum()}")
    print(f"   - 异常样本数：{df_merged_with_outlier['is_outlier'].sum()}")
    print(f"   - 校正后noise有效样本数：{df_final['corrected_noise'].notna().sum()}")
    if noise_stats.get('aidas'):
        aidas_stats = noise_stats['aidas']
        print(f"   - AIDAS组校正后noise均值：{aidas_stats.get('corrected_noise_均值', 'N/A')}")
        print(f"   - AIDAS组校正前后均值变化率：{aidas_stats.get('均值变化率(%)', 'N/A')}%")
    print("="*80)

# ---------------------- 运行入口 ----------------------
if __name__ == "__main__":
    # 运行前检查必要路径
    print("🔍 运行前检查路径配置...")
    required_paths = [
        FOLDER_PATH_AIDAS,
        BASELINE_TABLE_PATH  # 检查固定基线表格（含95分位数）
    ]
    
    # 检查文件夹/文件是否存在
    for path in required_paths:
        if not os.path.exists(path):
            print(f"❌ 错误：路径不存在 → {path}")
            print("请先修改代码顶部的路径配置，确保指向正确的文件夹/文件！")
            exit(1)
    
    # 检查元信息表（非必需，仅提醒）
    meta_paths = [META_AIDAS_MALE, META_AIDAS_FEMALE]
    missing_meta = [p for p in meta_paths if not os.path.exists(p)]
    if missing_meta:
        print(f"⚠️  警告：以下元信息表不存在（非必需，仅影响年龄/性别匹配）：")
        for p in missing_meta:
            print(f"   - {p}")
    
    # 执行主函数
    main()